<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 09 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Observe Sharding, Replication, and Node Failover</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Start an isolated 3 FE + 3 BE Doris cluster, trace logical tablets to physical replicas, observe execution across BE nodes, verify a Colocate Join, and test bounded BE and FE recovery.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 · Partition · Bucket · Tablet · Replica · Query Profile · Colocate Join · FE Election</span>
</div>

Run the cells in order. This Lab creates only `m09_*` objects in an isolated Module 9 cluster. It does not read the Level 1–8 container's private storage. Section 2 reads a bounded seven-hour scope from the same read-only AWS S3 course source used by Lab 1, then creates only `m09_*` objects in this isolated cluster.


### Initialize the Lab

Run the next cell before Section 1. It finds the course root whether the Notebook was opened from the course root or this module directory, loads the shared `DorisLab` helper, records the Module 9 Compose path, and configures this Notebook to use the Module 9 FE at MySQL port `9130` and HTTP port `8130`. It does **not** start Docker or change data.

Run it again after a kernel restart. The restart clears the Python connection object but leaves the Docker containers, named volumes, and imported tables unchanged. If the Module 9 cluster is still running, later SQL cells can reconnect automatically through port `9130`; the environment preparation and topology checks do not need to run again. A green **Lab tools are ready** message means the shared helper loaded successfully.

In [4]:
from pathlib import Path
import shlex
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

COMPOSE_FILE = COURSE_ROOT / "level3/module09-sharding-replication/compose.multinode.yml"
lab = DorisLab(
    lab_dir=COURSE_ROOT,
    connection_host="127.0.0.1",
    connection_port=9130,
    profile_http_host="127.0.0.1",
    profile_http_port=8130,
);


## 1. Start and identify the multi-node environment

Levels 1–8 use one integrated container. That environment can teach SQL, but it cannot show three physical copies of a Tablet, execution on several Backend (BE) nodes, or Frontend (FE) election.

Module 9 uses an isolated Compose project based on Apache Doris's official 4.1 multi-node file:

| Component | Count | Role in this Lab |
|---|---:|---|
| FE | 3 | Accept SQL, maintain metadata, elect one Master, and assign plan fragments. |
| BE | 3 | Store Tablet replicas and execute assigned plan fragments. |
| client | 1 | Wait until all six Doris nodes have registered; it also provides tools inside the Docker network. |

Allocate at least **4 CPU cores**, an **8 GB Docker memory setting**, and **20 GiB of free Docker data-disk space**. Run Module 9 without the Level 1–8 `doris` container so the two clusters do not compete for the same Docker memory pool. These are minimums for this small learning workload, not production sizing guidance. The course Compose uses a 512 MB heap limit for each FE and the official `BE_MEM_LIMIT=25%` setting for each BE. A percentage limit is a per-process ceiling; it does not reserve that memory for each BE.

All six Doris containers share one Docker memory pool. The three BE storage paths also use the same Docker Desktop or Docker Engine filesystem. Capacity reported three times by `SHOW BACKENDS` must therefore **not** be added together.

The project publishes FE MySQL ports `9130`–`9132`, FE HTTP ports `8130`–`8132`, and one BE HTTP port `8140` on localhost. Internal Doris ports stay unchanged.

The next cell follows Lab 1's initialization pattern: verify the host and Docker daemon, inspect resources and ports, validate the Compose file, reuse the pinned local image or pull it when absent, start the project with `--wait`, and show container health. It never stops or removes the Level 1–8 `doris` container. The preflight stops with a clear message when `doris` is running. Stop that container with `docker stop doris`, then rerun the cell; its named volumes preserve the earlier course data.

In [2]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}

echo "[1/5] Verify the host, Docker client, and daemon"
HOST_OS=$(uname -s)
HOST_ARCH=$(uname -m)
case "${{HOST_OS}}/${{HOST_ARCH}}" in
  Darwin/arm64|Darwin/aarch64|Linux/x86_64|Linux/amd64) ;;
  *) echo "Unsupported host: ${{HOST_OS}}/${{HOST_ARCH}}"; exit 2 ;;
esac
command -v docker >/dev/null 2>&1 || {{ echo 'Docker CLI was not found.'; exit 3; }}
docker info >/dev/null
docker compose version
docker info --format 'cpus={{{{.NCPU}}}} memory={{{{.MemTotal}}}} os={{{{.OperatingSystem}}}} arch={{{{.Architecture}}}}'

echo "[2/5] Inspect the isolated host ports"
if command -v lsof >/dev/null 2>&1; then
  lsof -nP -iTCP:8130 -iTCP:8131 -iTCP:8132 -iTCP:8140 -iTCP:9130 -iTCP:9131 -iTCP:9132 -sTCP:LISTEN || true
elif command -v ss >/dev/null 2>&1; then
  ss -ltn | grep -E ':(8130|8131|8132|8140|9130|9131|9132)\b' || true
fi

echo "[3/5] Validate the pinned course Compose"
docker compose -f "$COMPOSE_FILE" config --quiet
if [ "$(docker inspect -f '{{{{.State.Running}}}}' doris 2>/dev/null || true)" = "true" ]; then
  echo 'The Level 1-8 doris container is also running.'
  echo 'Stop it first with: docker stop doris'
  echo 'Its named volumes and course data are preserved.'
  exit 4
fi

echo "[4/5] Reuse or pull the pinned image, then start the isolated project"
if ! docker image inspect apache/doris:all-in-one-4.1.3 >/dev/null 2>&1; then
  docker pull apache/doris:all-in-one-4.1.3
else
  echo 'Pinned image already exists locally; skipping the registry pull.'
fi
docker compose -f "$COMPOSE_FILE" up -d --wait

echo "[5/5] Show Module 9 container health"
docker compose -f "$COMPOSE_FILE" ps
""", title="Prepare the Module 9 multi-node environment");


**Connect and verify the Doris topology and available resources**

Container health is only the first check. `SHOW FRONTENDS` verifies that three Follower-role FEs have joined and exactly one is the current Master. `SHOW BACKENDS` verifies that three BEs are alive and not decommissioned. Their `Version` columns identify the Doris build used by the experiment.

A BE can report `Alive=true` before it has reported a usable storage path, so the code waits for nonzero storage capacity. It then compares the current Docker allocation and shared data filesystem with the Lab minimums. The disk result uses the smallest capacity reported by the BEs as one shared filesystem value; it never multiplies that value by three.

In [3]:
lab.connect(
    container="doris-course-m09-fe-1-1",
    host="127.0.0.1",
    port=9130,
    http_port=8130,
)
lab.wait_for_node_state(
    "SHOW FRONTENDS",
    expected_total=3, expected_alive=3, expected_masters=1,
    expected_roles=("FOLLOWER",), require_same_version=True,
    title="Three joined FEs and the current Master",
)
lab.wait_for_node_state(
    "SHOW BACKENDS",
    expected_total=3, expected_alive=3, require_same_version=True,
    title="Three live BEs",
)
lab.environment_resource_summary(expected_backends=3);


Name,Host,Role,IsMaster,Join,Alive,Version
fe_e3887b98_f397_4c96_a237_893e38d56a94,172.31.91.41,FOLLOWER,false,true,true,doris-4.1.3-rc02-7126cf65d96
fe_19125cda_fb23_4dac_9640_5040476b4137,172.31.91.42,FOLLOWER,true,true,true,doris-4.1.3-rc02-7126cf65d96
fe_11a81f63_7872_4d49_9489_1c7d3825bd8e,172.31.91.40,FOLLOWER,false,true,true,doris-4.1.3-rc02-7126cf65d96


BackendId,Host,Alive,SystemDecommissioned,TabletNum,Version
1789884062978,172.31.91.50,true,false,241,doris-4.1.3-rc02-7126cf65d96
1789884062979,172.31.91.52,true,false,241,doris-4.1.3-rc02-7126cf65d96
1789884062980,172.31.91.51,true,false,240,doris-4.1.3-rc02-7126cf65d96


resource,current,Lab minimum,status,scope
Docker CPU allocation,8 cores,4 cores,PASS,shared by all FE and BE containers
Docker memory allocation,8.32 GB / 7.75 GiB,8.00 GB / 7.45 GiB reported,PASS,one shared memory pool; limits are not reservations
BE data filesystem: free,412.56 GiB,20 GiB,PASS,one shared Docker filesystem; do not add the three BE rows
BE data filesystem: total,452.13 GiB,informational,INFO,reported separately by each BE but backed by the same host


**Expected result**

- Seven Module 9 containers become healthy: `fe-1`–`fe-3`, `be-1`–`be-3`, and `client`.
- The FE result has three `Alive=true`, `Join=true` rows and exactly one `IsMaster=true` row.
- The BE result has three `Alive=true`, `SystemDecommissioned=false` rows, and each BE has reported a usable storage path.
- The resource table reports the current Docker CPUs, memory, and shared data-filesystem capacity beside the Lab minimum and a `PASS` or `BELOW MINIMUM` status. A below-minimum result stops the Lab before data is created.
- The displayed current values come from the learner's Docker Engine at runtime and therefore vary by computer. Only the Lab minimums are course requirements.
- The three BEs use the same Docker-host filesystem in this Compose environment. Capacity reported by each BE is repeated visibility into that shared filesystem and must not be added together.
- The locally verified image reports `doris-4.1.3-rc02-7126cf65d96`. A container count alone would not prove that the nodes joined Doris successfully; the SQL metadata and resource checks provide that evidence.

## 2. Load the course events, build shards, and verify three physical replicas

A **Partition** divides a table by a business range. Within each Partition, hash **Buckets** create logical **Tablets**. A Tablet is the unit Doris distributes among BEs. `replication_num = 3` asks Doris to keep three physical replicas of each logical Tablet on different BEs.

The Level 1 `events` table cannot be queried directly here because it belongs to the single-node container's private named volume. Instead, this section uses the same read-only AWS S3 course source as Lab 1 and imports the bounded interval **[2020-03-01 00:00:00, 2020-03-01 07:00:00)** into the isolated cluster. The verified seven-hour interval contains **12,062 events**. No S3 object is modified.

The same rows are stored in two Module 9 tables:

- `m09_events` hashes by `user_id`, which has many distinct values and should spread this sample reasonably evenly.
- `m09_events_skewed` hashes by `region`, which has only eight values. Six Buckets still exist in each Partition, but some Tablets can receive no rows while others receive many.

The seven-hour range is split into two time Partitions at 03:30. Both tables have two Partitions and six Buckets per Partition, so each has `2 × 6 = 12` logical Tablets. The setup creates these two Module 9 tables only when they are missing. Existing tables with the expected data can be reused on a rerun.

In [4]:
assert lab.load_s3_credentials(), "Add the read-only course S3 credentials before continuing."

lab.execute("CREATE DATABASE IF NOT EXISTS doris_course")
lab.execute("USE doris_course")

lab.execute("""
CREATE TABLE IF NOT EXISTS m09_events (
    event_time DATETIME,
    event_id BIGINT,
    user_id BIGINT,
    product_id BIGINT,
    event_type VARCHAR(16),
    region VARCHAR(16),
    revenue DECIMAL(12,2)
)
DUPLICATE KEY(event_time, event_id)
PARTITION BY RANGE(event_time) (
    PARTITION p_first_half VALUES LESS THAN ("2020-03-01 03:30:00"),
    PARTITION p_second_half VALUES LESS THAN ("2020-03-01 07:00:00")
)
DISTRIBUTED BY HASH(user_id) BUCKETS 6
PROPERTIES ("replication_num" = "3")
""")

lab.execute("""
CREATE TABLE IF NOT EXISTS m09_events_skewed (
    event_time DATETIME,
    event_id BIGINT,
    user_id BIGINT,
    product_id BIGINT,
    event_type VARCHAR(16),
    region VARCHAR(16),
    revenue DECIMAL(12,2)
)
DUPLICATE KEY(event_time, event_id)
PARTITION BY RANGE(event_time) (
    PARTITION p_first_half VALUES LESS THAN ("2020-03-01 03:30:00"),
    PARTITION p_second_half VALUES LESS THAN ("2020-03-01 07:00:00")
)
DISTRIBUTED BY HASH(region) BUCKETS 6
PROPERTIES ("replication_num" = "3")
""");

In [5]:
lab.load_table_once(
    "m09_events",
    """
    INSERT INTO m09_events
    SELECT
        event_time,
        event_id,
        user_id,
        product_id,
        event_type,
        region,
        revenue
    FROM {{COURSE_S3}}
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-01 07:00:00'
    """,
    title="Prepare the bounded course events",
    low_memory_s3=True,
    expected_rows=12_062,
)
lab.load_table_once(
    "m09_events_skewed",
    "INSERT INTO m09_events_skewed SELECT * FROM m09_events",
    title="Prepare the low-cardinality distribution comparison",
    expected_rows=12_062,
);

The load cell checks each target before writing. A fresh table is populated with one atomic `INSERT INTO SELECT`; a table with the previously committed 12,062-row course load is reused without another INSERT. The helper records success through committed Partition versions and also rejects a different metadata row count once that asynchronous count is available, instead of silently mixing datasets. On the first run, both cards report 12,062 affected rows. On a rerun, both cards report that the existing rows were reused. The next cell first checks each table once: a missing, empty, or incorrectly sized table fails immediately with an instruction to run the insert cell. Only the Tablet row counts are allowed to wait, because BEs report that storage metadata asynchronously. The wait is bounded at three minutes and polls every two seconds. `Rows / tablet` is a storage metadata range, not an additional SQL result count.

In [6]:
lab.wait_for_tablet_rows("m09_events", expected_rows=12062)
lab.wait_for_tablet_rows("m09_events_skewed", expected_rows=12062)
lab.storage_layout_summary(["m09_events", "m09_events_skewed"]);


table,partitions,buckets / partition,tablets,replicas / tablet,logical rows,rows / tablet,local replica size,version count
m09_events,2,6,12,3,"12,062","480–1,585",587.39 KB,2
m09_events_skewed,2,6,12,3,"12,062","0–3,321",544.91 KB,2


**Expected result**

Read the shared structure first:

- `partitions = 2` means that the seven-hour interval is divided into two time ranges.
- `buckets / partition = 6` means that Doris hashes the rows in each Partition into six Buckets.
- Each Bucket is stored as one logical Tablet, so `2 × 6 = 12` Tablets for each table.
- `replicas / tablet = 3` means that Doris stores three physical copies of every logical Tablet on the three BEs. Each table therefore has 12 logical Tablets and 36 physical replicas.
- `logical rows = 12,062` for both tables. Replication protects the same logical rows; it does not make SQL return three copies of every event.

Now compare `rows / tablet`, which is the main evidence in this table:

- `m09_events` hashes by the high-cardinality `user_id`. Its Tablet counts range from 480 to 1,585, and no Tablet is empty. The sample is spread across all 12 Tablets.
- `m09_events_skewed` hashes by `region`, which has only eight distinct values. Its range is 0 to 3,321: at least one Tablet is empty while another contains many more rows. This wider range is evidence of **data skew** in this sample.

The final two columns are supporting storage metadata:

- `local replica size` adds the compressed local size reported for all physical replicas of the table. The exact number can vary after another run or Compaction. A smaller value here does not mean that the distribution key is better.
- `version count` is the number of stored data versions reported for each physical replica. It is not the replica count. The value can change after further writes or Compaction and is not used to judge distribution quality in this experiment.

The exact row ranges and storage sizes are observations from this bounded sample. The conclusion is based on the shape of the distribution: `user_id` uses every Tablet with a narrower range, while low-cardinality `region` leaves empty Tablets and concentrates more rows in others.

Next, verify all 12 logical Tablets without printing 12 repetitive metadata rows. The helper checks every Tablet, then presents one summary row. `BE nodes / tablet = 3` confirms that each Tablet’s three physical replicas are placed on three distinct BEs.

In [7]:
lab.tablet_replica_health_summary(
    "m09_events",
    expected_tablets=12,
    expected_replicas=3,
    expected_alive=3,
    expected_complete=3,
    title="Replica health across all m09_events Tablets",
);


table,tablets,replicas / tablet,BE nodes / tablet,alive replicas / tablet,version-complete / tablet
m09_events,12,3,3,3,3


**Expected result**

- On the first run, each atomic insert reports 12,062 affected rows. On a rerun, the helper reports 12,062 reused rows and confirms that no new INSERT ran.
- Replication does not turn 12,062 logical rows into three times as many query rows.
- Each table has two Partitions, six Buckets per Partition, 12 unique Tablets, and three replica records per Tablet.
- The compact replica-health row summarizes all 12 Tablets: each has three replicas on three BE nodes, and all three replicas are alive and version-complete.
- The two tables contain identical logical rows. Their different rows-per-Tablet ranges come from hashing a high-cardinality `user_id` versus an eight-value `region`.

## 3. Trace and compare one query across BE nodes

The FE creates a distributed plan and assigns scan instances to BEs. `EXPLAIN` describes that plan: the scan covers 12 Tablets on three nodes, partial aggregates run near the scans, and Exchange operators move intermediate rows for final aggregation.

The Query Profile records what actually ran. The helper temporarily enables Profile level 2 and disables SQL Cache, Query Cache, and Condition Cache so each run produces fresh operator evidence; it restores those session settings afterward. Storage, operating-system, and index caches are not flushed, so this is not a cold-cache timing benchmark.

First run the regional aggregate on `m09_events`, whose high-cardinality `user_id` distribution used every Tablet. Then run the same SQL on `m09_events_skewed`, whose eight-value `region` distribution produced empty and oversized Tablets in Section 2. Both captures use the same pipeline parallelism so their individual scan-task `ScanRows` values can be compared. Keep both SQL statements visible and retain cropped excerpts from the original `DetailProfile`; do not compare elapsed time.

In [8]:
regional_query = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m09_events
GROUP BY region
ORDER BY region
"""
regional_evidence = lab.capture_query(
    regional_query, settings={"parallel_pipeline_task_num": "4"}
)
regional_evidence.show_result("Regional aggregate result")
regional_evidence.show_distributed_plan("Plan fragments, Exchanges, and scan scope")
regional_evidence.show_distributed_scan_profile("Actual scan tasks from DetailProfile")

scan_work = regional_evidence.distributed_scans()
assert scan_work["be_host"].nunique() == 3
assert int(scan_work["selected_tablets"].sum()) == 12
assert int(regional_evidence.rows["event_count"].sum()) == 12062;


region,event_count,total_revenue
region_01,1557,166124.28
region_02,1553,195049.05
region_03,1458,182170.95
region_04,1354,158312.41
region_05,1544,168723.56
region_06,1588,216008.69
region_07,1608,185378.93
region_08,1400,162868.82


In [9]:
skewed_regional_query = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m09_events_skewed
GROUP BY region
ORDER BY region
"""
skewed_evidence = lab.capture_query(
    skewed_regional_query, settings={"parallel_pipeline_task_num": "4"}
)
skewed_evidence.show_result("Same regional aggregate from the skewed layout")
lab.assert_same_rows(regional_evidence.rows, skewed_evidence.rows)
skewed_evidence.show_distributed_scan_profile(
    "Actual scan tasks from the low-cardinality layout"
)

skewed_work = skewed_evidence.distributed_scans()
assert skewed_work["be_host"].nunique() == 3
assert int(skewed_work["selected_tablets"].sum()) == 12

balanced_task_rows = regional_evidence.distributed_scan_tasks()["scan_rows"].dropna().astype(int)
skewed_task_rows = skewed_evidence.distributed_scan_tasks()["scan_rows"].dropna().astype(int)
assert skewed_task_rows.max() - skewed_task_rows.min() > balanced_task_rows.max() - balanced_task_rows.min();

region,event_count,total_revenue
region_01,1557,166124.28
region_02,1553,195049.05
region_03,1458,182170.95
region_04,1354,158312.41
region_05,1544,168723.56
region_06,1588,216008.69
region_07,1608,185378.93
region_08,1400,162868.82


**Expected result**

Read the output in four passes.

**1. Check that logical results do not depend on physical distribution**

Both result tables contain the same eight regions with identical counts and revenue. `assert_same_rows` checks the complete result, not only its total. Changing the Distribution Key changes physical placement, not the logical rows stored by the table.

**2. Read the `m09_events` EXPLAIN excerpt from `PLAN FRAGMENT 2` upward**

```text
FROM m09_events             -> VOlapScanNode / TABLE: ...m09_events
no WHERE clause             -> partitions=2/2, tablets=12/12
GROUP BY region             -> VAGGREGATE (update serialize)
same region must meet       -> HASH_PARTITIONED: region / EXCHANGE ID: 02
finish COUNT and SUM        -> VAGGREGATE (merge finalize)
ORDER BY region             -> VSORT
return one result stream    -> EXCHANGE ID: 05 / VMERGING-EXCHANGE / VRESULT SINK
```

`numNodes=3` says the FE planned scan work on three nodes. This remains planned evidence; use the Profile excerpts to see what actually ran.

**3. Read each cropped `DetailProfile` by BE and scan task**

```text
Pipeline ... hostname:<one BE>
  PipelineTask(index=...)
    OLAP_SCAN_OPERATOR(...table name...)
      TabletIds: [...]   <- Tablet replicas selected by this task
      ScanRows: ...      <- rows this task actually scanned
```

Each Profile must contain three BE hosts and 12 distinct Tablet IDs. A scan operator may contain more than one Tablet ID, so the number of `PipelineTask` blocks need not equal 12.

**4. Compare the shapes of the two task workloads**

In the `m09_events` Profile, the high-cardinality `user_id` distribution produces a narrower spread of `ScanRows` among tasks. In the `m09_events_skewed` Profile, the low-cardinality `region` distribution produces a wider spread; a task can scan an empty Tablet while another handles a much larger Tablet. The assertion compares the observed ranges from these two fresh Profiles without requiring fixed row counts, task indexes, Tablet IDs, or BE assignments.

This demonstrates a runtime consequence of data skew: parallel scan tasks can receive unequal amounts of work even though the final SQL answers are identical. It does not prove a fixed latency penalty. All BEs share one Docker host, and caches, task scheduling, aggregation, and other operators also affect elapsed time. The complete EXPLAIN and Query Profiles remain folded below the cropped evidence.

## 4. Reuse `m09_events` placement with a Colocate Join

The imported event table is already distributed by `HASH(user_id)` with six Buckets and three replicas. Reuse that design for a realistic enrichment query: attach a small user segment to each event by joining on `user_id`.

Doris allows an existing compatible table to join a Colocation Group. The first cell adds `m09_events` to `m09_user_group` without copying its rows, then defines one new user dimension with the same `BIGINT` distribution key, Bucket count, replica allocation, and group. The dimension uses one row per user from the existing bounded event scope.

As in Section 2, setup is divided into definitions, idempotent loading, and metadata verification. `CREATE TABLE IF NOT EXISTS` preserves an existing dimension, while `load_table_once` inserts only when that table is new. Group membership and `IsStable=true` prove that the placement contract is currently stable. They do not yet prove that the optimizer selected a Colocate Join; the following query plan supplies that evidence.

In [ ]:
lab.execute("""
ALTER TABLE m09_events
SET ("colocate_with" = "m09_user_group")
""")

lab.execute("""
CREATE TABLE IF NOT EXISTS m09_users (
    user_id BIGINT,
    user_segment VARCHAR(32)
)
UNIQUE KEY(user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 6
PROPERTIES (
    "replication_num" = "3",
    "colocate_with" = "m09_user_group"
)
""");

In [ ]:
lab.load_table_once(
    "m09_users",
    """
    INSERT INTO m09_users
    SELECT DISTINCT
        user_id,
        CONCAT('segment_', user_id % 4 + 1)
    FROM m09_events
    """,
    title="Prepare the colocated user dimension",
    expected_rows=6_706,
);

In [ ]:
lab.wait_for_tablet_rows("m09_users", expected_rows=6_706)
colocation_groups = lab.sql(
    "SHOW PROC '/colocation_group'",
    title="Stable Module 9 Colocation Group",
    columns=["GroupName", "BucketsNum", "ReplicaAllocation", "DistCols", "IsStable"],
)
user_group = colocation_groups[
    colocation_groups["GroupName"].str.endswith(".m09_user_group")
]
assert len(user_group) == 1
assert str(user_group.iloc[0]["BucketsNum"]) == "6"
assert str(user_group.iloc[0]["ReplicaAllocation"]) == "tag.location.default: 3"
assert str(user_group.iloc[0]["DistCols"]).lower() == "bigint"
assert str(user_group.iloc[0]["IsStable"]).lower() == "true";

In [ ]:
colocate_query = """
SELECT u.user_segment, COUNT(*) AS event_count, SUM(e.revenue) AS total_revenue
FROM m09_events e
JOIN m09_users u
  ON e.user_id = u.user_id
WHERE e.event_type = 'purchase'
GROUP BY u.user_segment
ORDER BY u.user_segment
"""
colocate_result = lab.sql(colocate_query, title="Purchase totals by user segment")
colocate_plan = lab.explain_colocate_join(
    "EXPLAIN " + colocate_query,
    title="Selected Colocate Join and its two scans",
)
assert int(colocate_result["event_count"].sum()) == 4_809;

**Expected result**

- `ALTER TABLE` adds the existing `m09_events` table to `m09_user_group`; it does not create another event table or reload the 12,062 events.
- On the first run, the load cell creates 6,706 user rows derived from the bounded event scope. On a rerun, its card reports that the existing rows were reused, and no duplicate INSERT is performed.
- `SHOW PROC '/colocation_group'` contains one `m09_user_group` row with six Buckets, three replicas, `BIGINT` distribution columns, and `IsStable=true`.
- The query result contains four user segments whose event counts sum to 4,809 purchases.
- The plan excerpt contains `INNER JOIN(COLOCATE...)`, the equality `user_id = user_id`, and scans of both `m09_events` and `m09_users`. This proves that the Join itself reuses their compatible `user_id` placement.
- The later aggregation groups by `user_segment`, rather than the `user_id` distribution key, so another Exchange may still appear after the local Join.
- No timing comparison is made: all six Doris nodes share one Docker host, CPU pool, memory pool, and physical storage.

## 5. Observe one BE stopping and returning

A Tablet with three replicas can remain readable when one BE stops because two physical copies are still alive. The exact outcome of an in-flight request can vary, so this experiment waits until FE metadata reports the node down before issuing bounded probes.

The existing Module 9 tables already have three replicas, so `m09_events` is reused to verify that reads remain complete during the outage. A separate one-row Unique Key probe is needed only for the write-and-recovery evidence: writing into the Duplicate Key event tables would change their fixed 12,062-row baseline, while changing `m09_users` would mutate the dimension created by Section 4. The probe lets the Lab create one known version while a BE is absent and then verify that the returning replica catches up, without changing earlier results. Reusing `probe_id = 1` keeps one logical probe row if the write cell is repeated. A normal `SELECT` reads the logical table and does not expose three separate physical copies, so replica versions provide the per-BE evidence.

First create the probe and commit its initial row. The next cell checks replica versions rather than waiting for asynchronous Tablet row-count statistics: Section 5 needs to establish that all three physical copies are current before stopping a BE.

Then stop only `be-3`; do not use `docker compose down`, because `down` would remove the entire isolated cluster.


In [19]:
lab.execute("DROP TABLE IF EXISTS m09_availability_probe")
lab.execute("""
CREATE TABLE m09_availability_probe (
    probe_id BIGINT,
    note VARCHAR(64)
)
UNIQUE KEY(probe_id)
DISTRIBUTED BY HASH(probe_id) BUCKETS 3
PROPERTIES (
    "replication_num" = "3",
    "enable_unique_key_merge_on_write" = "true"
)
""")
lab.insert(
    "INSERT INTO m09_availability_probe VALUES (1, 'all three BEs alive')",
    title="Create the initial availability probe",
    expected_rows=1,
)
lab.sql("""
SELECT probe_id, note
FROM m09_availability_probe
ORDER BY probe_id
""", title="Initial availability probe row");


probe_id,note
1,all three BEs alive


The insert card reports one affected row, and the result table shows what that row means: `probe_id = 1` is the reusable key and `note = 'all three BEs alive'` is its initial value. Now wait only until every probe Tablet has three live, current-version replicas. This is the baseline needed for the failure experiment.


In [20]:
lab.wait_for_replica_health(
    "m09_availability_probe", expected_alive=3, expected_complete=3,
)
lab.tablet_replica_health_summary(
    "m09_availability_probe",
    title="Probe replicas before the BE stop",
    expected_tablets=3,
    expected_replicas=3,
    expected_alive=3,
    expected_complete=3,
);


table,tablets,replicas / tablet,BE nodes / tablet,alive replicas / tablet,version-complete / tablet
m09_availability_probe,3,3,3,3,3


In [21]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" stop be-3
""", title="Stop only be-3");

lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=2,
    title="FE metadata after be-3 stops",
)
lab.tablet_replica_health_summary(
    "m09_events", title="Stored and alive m09_events replicas during the outage"
);


 Container doris-course-m09-be-3-1 Stopping 
 Container doris-course-m09-be-3-1 Stopped 


BackendId,Host,Alive,SystemDecommissioned,TabletNum,Version
1789884062978,172.31.91.50,true,false,253,doris-4.1.3-rc02-7126cf65d96
1789884062979,172.31.91.52,false,false,253,doris-4.1.3-rc02-7126cf65d96
1789884062980,172.31.91.51,true,false,252,doris-4.1.3-rc02-7126cf65d96


table,tablets,replicas / tablet,BE nodes / tablet,alive replicas / tablet,version-complete / tablet
m09_events,12,3,3,2,3


In [2]:
lab.sql("""
SELECT COUNT(*) AS logical_rows, SUM(revenue) AS total_revenue
FROM m09_events
""", title="Read using the remaining replicas")

lab.execute("""
INSERT INTO m09_availability_probe VALUES (1, 'write committed with one BE stopped')
""")
lab.sql("""
SELECT probe_id, note
FROM m09_availability_probe
ORDER BY probe_id
""", title="Idempotent write probe while one BE is stopped")

replicas_during_outage = lab.tablet_replica_backend_summary(
    "m09_availability_probe",
    title="Which BEs contain the current probe version during the outage",
)
assert replicas_during_outage["alive"].sum() == 2
assert sorted(replicas_during_outage["current-version replicas"].tolist()) == [0, 3, 3];


logical_rows,total_revenue
12062,1434636.69


probe_id,note
1,write committed with one BE stopped


backend_id,alive,stored replicas,current-version replicas,replica versions
1789884062978,True,3,3,4
1789884062979,False,3,0,2
1789884062980,True,3,3,4


The `SELECT` result is the one logical answer Doris serves from the available replicas. The three-row version table is the physical evidence: all three BEs still own replica records, but the stopped BE is not alive and has zero replicas at the newest committed version. The two live BEs each have the newest version for all three probe Tablets.

The read and write above are observations from this configured three-replica cluster after FE marked one BE unavailable. They do not promise that every request has zero interruption: a request already assigned to the stopped node may fail and require a client retry.

Start the same container again. Capturing an intermediate state immediately after `docker compose start` would be unreliable because the small probe can catch up before the next statement runs. Instead, the helper waits for all three replicas to reach the current version and then prints the same per-BE table again.


In [3]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" start be-3
""", title="Restart be-3");

lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=3,
    title="All three BEs are alive again",
)
lab.wait_for_replica_health(
    "m09_availability_probe", expected_alive=3, expected_complete=3,
)
replicas_after_recovery = lab.tablet_replica_backend_summary(
    "m09_availability_probe",
    title="All BEs contain the current probe version after recovery",
)
assert replicas_after_recovery["alive"].all()
assert (replicas_after_recovery["current-version replicas"] == 3).all();


 Container doris-course-m09-fe-1-1 Waiting 
 Container doris-course-m09-fe-1-1 Healthy 
 Container doris-course-m09-be-3-1 Starting 
 Container doris-course-m09-be-3-1 Started 


BackendId,Host,Alive,SystemDecommissioned,TabletNum,Version
1789884062978,172.31.91.50,true,false,253,doris-4.1.3-rc02-7126cf65d96
1789884062979,172.31.91.52,true,false,253,doris-4.1.3-rc02-7126cf65d96
1789884062980,172.31.91.51,true,false,252,doris-4.1.3-rc02-7126cf65d96


backend_id,alive,stored replicas,current-version replicas,replica versions
1789884062978,True,3,3,4
1789884062979,True,3,3,4
1789884062980,True,3,3,4


**Expected result**

- The first card reports one affected row immediately after the initial probe write. The table below it shows one logical row: `probe_id = 1` and `note = 'all three BEs alive'`. The value `1` is the number of rows written, not the replica count.
- Before the stop, the replica-health summary reports three Tablets, three replicas per Tablet, and three current-version replicas per Tablet.
- During the stop, `SHOW BACKENDS` contains three registered BEs but only two are alive. The `m09_events` aggregate still returns 12,062 logical rows.
- The second probe write replaces the value for `probe_id = 1` with `write committed with one BE stopped`; it does not append a second logical row.
- In the per-BE version table during the outage, two live BEs each report three current-version probe replicas. The stopped BE still owns three stored replica records but reports `alive = false` and zero current-version replicas. This is the evidence that it missed the committed version while offline.
- After restart and version catch-up, the same table reports all three BEs alive and three current-version replicas on every BE. The logical value remains `write committed with one BE stopped`; recovery copies that committed version to the returning replica.
- The Lab deliberately does not try to capture a partly recovered state immediately after `docker compose start`, because this one-row probe may catch up too quickly for that state to be reproducible. This environment has no fourth BE, so the experiment does not demonstrate replacement-replica migration or decommissioning.


## 6. Observe FE election and reconnect through another FE

BE replicas protect table data. FE Followers protect the metadata service through quorum and Master election. They solve a different problem.

The code first discovers the current Master from `SHOW FRONTENDS`; it does not assume `fe-1` is still Master after an earlier run. It maps the internal FE address to this Lab's Compose service and published localhost ports, stops that service, reconnects through a surviving FE, and waits for one of the two survivors to become Master.

The old MySQL connection is not a load balancer. Successful election and successful client reconnection are therefore checked separately.


In [15]:
fe_endpoints = {
    "172.31.91.40": {"service": "fe-1", "port": 9130, "http_port": 8130},
    "172.31.91.41": {"service": "fe-2", "port": 9131, "http_port": 8131},
    "172.31.91.42": {"service": "fe-3", "port": 9132, "http_port": 8132},
}
fe_state_before = lab.sql("SHOW FRONTENDS", title="FE state before stopping the Master")
master_host = str(fe_state_before.loc[fe_state_before["IsMaster"] == "true", "Host"].iloc[0])
master_endpoint = fe_endpoints[master_host]
survivor_host = next(host for host in fe_endpoints if host != master_host)
survivor_endpoint = fe_endpoints[survivor_host]
master_service = master_endpoint["service"]

lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" stop {shlex.quote(master_service)}
""", title=f"Stop the current Master service: {master_service}");


Name,Host,Role,IsMaster,Join,Alive,Version
fe_e3887b98_f397_4c96_a237_893e38d56a94,172.31.91.41,FOLLOWER,false,true,true,doris-4.1.3-rc02-7126cf65d96
fe_19125cda_fb23_4dac_9640_5040476b4137,172.31.91.42,FOLLOWER,true,true,true,doris-4.1.3-rc02-7126cf65d96
fe_11a81f63_7872_4d49_9489_1c7d3825bd8e,172.31.91.40,FOLLOWER,false,true,true,doris-4.1.3-rc02-7126cf65d96


 Container doris-course-m09-fe-3-1 Stopping 
 Container doris-course-m09-fe-3-1 Stopped 


In [16]:
survivor_service = survivor_endpoint["service"]
lab.connect(
    container=f"doris-course-m09-{survivor_service}-1",
    host="127.0.0.1",
    port=survivor_endpoint["port"],
    http_port=survivor_endpoint["http_port"],
)
lab.execute("USE doris_course")
lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=2, expected_masters=1,
    title="Two live FEs and a newly elected Master",
)
lab.sql("""
SELECT COUNT(*) AS logical_rows
FROM m09_events
""", title="Read after reconnecting through a surviving FE");


Name,Host,Role,IsMaster,Join,Alive,Version
fe_7bc374c3_d857_4b91_9670_de9cbb3b9775,172.31.91.40,FOLLOWER,false,true,true,doris-4.1.3-rc02-7126cf65d96
fe_b0df5629_2311_49a0_9fe6_5fad88a42db9,172.31.91.41,FOLLOWER,true,true,true,doris-4.1.3-rc02-7126cf65d96
fe_5a0ec877_c98e_417e_ae35_24af1b1ba4ca,172.31.91.42,FOLLOWER,false,true,false,NULL


logical_rows
12062


Restart the former Master and keep querying through the surviving FE endpoint. The returned node should rejoin as an alive Follower; Doris does not need to hand leadership back to it.


In [17]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" start {shlex.quote(master_service)}
""", title=f"Restart the former Master service: {master_service}");

lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=3, expected_masters=1,
    title="All three FEs after the former Master rejoins",
);


 Container doris-course-m09-fe-1-1 Waiting 


 Container doris-course-m09-fe-1-1 Healthy 


 Container doris-course-m09-fe-3-1 Starting 


 Container doris-course-m09-fe-3-1 Started 


Name,Host,Role,IsMaster,Join,Alive,Version
fe_7bc374c3_d857_4b91_9670_de9cbb3b9775,172.31.91.40,FOLLOWER,false,true,true,doris-4.1.3-rc02-7126cf65d96
fe_b0df5629_2311_49a0_9fe6_5fad88a42db9,172.31.91.41,FOLLOWER,true,true,true,doris-4.1.3-rc02-7126cf65d96
fe_5a0ec877_c98e_417e_ae35_24af1b1ba4ca,172.31.91.42,FOLLOWER,false,true,true,doris-4.1.3-rc02-7126cf65d96


**Expected result**

- After the stop, two FEs are alive and exactly one of them reports `IsMaster=true`.
- Reconnecting through another published FE port allows the 12,062-row read to succeed.
- After restart, all three FEs are alive and only one is Master. The former Master may remain a non-Master Follower.
- The BE nodes and table replicas were not stopped by the FE election. All six Doris nodes still share one Docker host, Docker Engine, disk, and external network, so this is container-level behavior rather than proof of host-, rack-, or availability-zone-level high availability.


### Stop the Module 9 cluster

Run this optional cell when you have finished working and want to release the six Doris nodes' CPU and memory. It stops the existing Compose containers but does not remove them or their writable layers, so the Module 9 metadata and `m09_*` tables remain available. Skip it if you want to continue using the cluster.


In [5]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}

docker compose -f "$COMPOSE_FILE" stop
docker compose -f "$COMPOSE_FILE" ps -a
""", title="Stop the Module 9 cluster");


 Container doris-course-m09-client-1 Stopping 
 Container doris-course-m09-client-1 Stopped 
 Container doris-course-m09-be-1-1 Stopping 
 Container doris-course-m09-be-3-1 Stopping 
 Container doris-course-m09-be-2-1 Stopping 
 Container doris-course-m09-fe-2-1 Stopping 
 Container doris-course-m09-fe-3-1 Stopping 
 Container doris-course-m09-fe-3-1 Stopped 
 Container doris-course-m09-fe-2-1 Stopped 
 Container doris-course-m09-be-3-1 Stopped 
 Container doris-course-m09-be-1-1 Stopped 
 Container doris-course-m09-be-2-1 Stopped 
 Container doris-course-m09-fe-1-1 Stopping 
 Container doris-course-m09-fe-1-1 Stopped 
NAME                        IMAGE                           COMMAND                  SERVICE   CREATED        STATUS                              PORTS
doris-course-m09-be-1-1     apache/doris:all-in-one-4.1.3   "/usr/bin/tini -- /o…"   be-1      22 hours ago   Exited (0) 2 seconds ago            
doris-course-m09-be-2-1     apache/doris:all-in-one-4.1.3   "/usr/bin/tini

**Expected result:** every `doris-course-m09` service reports `Exited`. Unlike `docker compose down`, this operation keeps the containers and their internal FE metadata and BE data.

### Restart the Module 9 cluster

Run this cell when you want to continue with the same Module 9 tables. It makes sure Docker is available, starts the existing Compose containers, reconnects through `fe-1`, verifies that all three FEs and all three BEs rejoined, and reads the persisted event table.

This restart cell is idempotent: it can be run when Docker Desktop and the Compose containers are stopped, starting, or already running. It deliberately uses `docker compose start`, not `up` or `down`, so a missing cluster fails instead of silently creating an empty replacement.


In [ ]:
lab.ensure_docker_ready()
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}

docker compose -f "$COMPOSE_FILE" start
docker compose -f "$COMPOSE_FILE" ps -a
""", title="Restart the Module 9 cluster")

lab.connect(
    container="doris-course-m09-fe-1-1",
    host="127.0.0.1",
    port=9130,
    http_port=8130,
)
lab.execute("USE doris_course")
lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=3, expected_masters=1,
    title="Recovered three-FE topology",
)
lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=3,
    title="Recovered three-BE topology",
)
lab.sql("""
SELECT COUNT(*) AS recovered_rows
FROM m09_events
""", title="Recovered Module 9 event table");


**Expected result:** all three FEs and all three BEs report `Alive=true`, exactly one FE reports `IsMaster=true`, and `m09_events` still contains **12,062** rows. This verifies recovery of the stopped containers and their existing Module 9 data.



## What you proved

- Two Partitions with six Buckets each produced 12 logical Tablets; three replicas produced three physical copies of every Tablet without tripling SQL results.
- A high-cardinality hash key spread this sample more evenly than an eight-value key.
- `EXPLAIN` showed a distributed plan, while `DetailProfile` showed actual scan work on all three BEs and exactly 12,062 scanned logical rows.
- Compatible and stable colocated layouts allowed the optimizer to select a Colocate Join.
- After one BE stopped, metadata distinguished three stored replica records from two currently alive copies; after restart, heartbeat and replica-version recovery were checked separately.
- After the current Master FE stopped, the remaining Followers elected a Master, but the client still had to reconnect through a live FE endpoint.

Official references: [All-in-One multi-node image](https://doris.apache.org/community/developer-guide/all-in-one-image) · [System architecture](https://doris.apache.org/docs/4.x/features-architecture/system-architecture/) · [Replica management](https://doris.apache.org/docs/4.x/admin-manual/maint-monitor/tablet-repair-and-balance/) · [Colocation Join](https://doris.apache.org/docs/4.x/query-acceleration/colocation-join/)
